# Customer Churn Prediction System (TensorFlow)
## Projenin Amacı:
Bir müştrenin şirketten ayrılıp ayrılmayacağını (churn) tahmin eden bir sinir ağı kurmaktır. Binary classification problemidir.

## Tensorflow neyi çözer?
TensorFlow, sinir ağlarını (neural network) kurmak, eğitmek ve değerlendirmek için kullanılan bir kütüphanedir.

## Kapsanan Tensorflow konuları:
- TensorFlow giriş (tensor mantığı)
- Katmanlar (Dense)
- Aktivasyon fonksiyonları
- Regresyon vs Sınıflandırma farkı
- Train / Validation / Test setleri
- TensorFlow ile sınıflandırma örneği


## 2- Veri Oluşturma (Numpy)

In [15]:
import numpy as np
import pandas as pd
import tensorflow as tf

#Numpy ile sentetik müşteri verisi oluşturalım 
#Feature matrisi X, target ise y olacak
#Müşteri için elimizde yaş, aylık harcama, abonelik süresi, destek talep sayısı olacak
#churn (0 kalır, 1 ayrılır)

num = 2400 #2400 adet
age = np.random.randint(18, 75, size = num)   #18 yaşından -74'e kadar 2400 adet
monthly_spend = np.random.uniform(1000, 10000, size = num) #aylık harcama tl
subscription = np.random.randint(1, 37, size=num) #abonelik süresi 1-36 ay
#poisson dağılımı -> belirli zaman diliminde kaç olay gerçekleşir 
#kaç kez destek talebi açmış müşteri
#lam -> ortalama olay sayısı yani destek talep sayısı bizim için
support_request = np.random.poisson(lam = 2.0, size = num ) #ort 2 talep sayısı

#Feature matrisi - np.column_stack() dizileri sütun olarak birleştirir
#tensorflow genelde float32 veri tipi ister
X = np.column_stack([age, monthly_spend, subscription, support_request]).astype(np.float32)

#Target (y)
#abonelik süresi uzadıkça ayrılma azalır.
#harcama çok yüksekse bazen ayrılma düşebilir
# destek talep artarsa ayrılma artsın
#müşterinin ayrılma eğilimi, c_prob büyürse ayrılma ihtimali (churn) artar
c_prob = (
    #müşteri çok destek talebi açıyorsa memnuniyetsiz olabilir
    0.9 * support_request 
    - 1.2 * (subscription / 36) #etkisini dengeledik
    - 0.6 * (monthly_spend / 6000) #çok harcayan müşteri genelde hizmetten memnundur. 1000'e bölme nedenimiz monthly spend yüksek olduğu için dengeli dağılsın diye
    + 0.05 * ((age - 40) / 10)
)

#skor -> olasılığı (1-0)
prob = 1 / (1 + np.exp(-c_prob))

#olasılık -> 1/0 churn etkisi
y = (np.random.rand(num) > prob).astype(np.int32)

print("X: ", X.shape)
print("y: ", y.shape)
print("Churn rate: ", y.mean().round(2))

X:  (2400, 4)
y:  (2400,)
Churn rate:  0.38


## 3- Train-Test Ayırma

In [5]:
!pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 13.5 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 16.0 MB/s  0:00:01m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [6]:
import sklearn
print(sklearn.__version__)

1.8.0


In [16]:
#Train: modelin öğrenmesi, %70
#validation: model ayarlarını kontrol etmek %15
# test ise gerçek performansı ölçmektir %15

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.30, random_state=42)
#validation ve test olarak bölelim (15-15)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Train set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}" )

Train set: (1680, 4)
Validation set: (360, 4)
Test set: (360, 4)


## 4–TensorFlow Modeli Kurma

In [17]:
#Sequential model oluşturulacak
# verimiz 4 featuredan oluşuyor yani input layer 4
# inpıt (4) -> dense - hidden layer (16) -> dense - hidden layer(8) -> dense - output (1)  
#Dense -> fully connected (tam bağlantılı) katmanı ifade eder. Dense katmanında, katmandaki her bir nöron, kendisinden önceki katmanda bulunan tüm nöronlardan girdi alır.
#Aktivasyon fonksiyonu -> ReLU
#Output layer -> sigmoid (1-0)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    #hidden layer 1
    Dense(16,  input_shape = (4,)),

    #hidden layer 2
    Dense(8),

    #output layer
    Dense(1)
    
])
model.summary() #model yapısı

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 16)                80        
                                                                 
 dense_10 (Dense)            (None, 8)                 136       
                                                                 
 dense_11 (Dense)            (None, 1)                 9         
                                                                 
Total params: 225 (900.00 Byte)
Trainable params: 225 (900.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## 5– Aktivasyon Fonksiyonları

In [18]:
model = Sequential([
    #hidden layer 1
    Dense(16, activation="relu", input_shape=(4,)),

    #hidden layer 2
    Dense(8, activation="relu"),

    #output layer
    Dense(1, activation="sigmoid")
])

model.summary()

#Neden sigmoid? : Sigmoid kullandık çünkü bu bir ikili sınıflandırma (1-0) problemidir.
#sigmoid çıktıyı 0 ile 1 arasında bir olasılığa dönüştürür.

#Neden ReLU? : Sinir ağlarında en yaygın ve en hızlı çalışan aktivasyon fonksiyonlarındandır.
#negatif değerleri sıfır yapar ve modelin doğrusal olmayan ilişkileri öğrenmesini sağlar

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_12 (Dense)            (None, 16)                80        
                                                                 
 dense_13 (Dense)            (None, 8)                 136       
                                                                 
 dense_14 (Dense)            (None, 1)                 9         
                                                                 
Total params: 225 (900.00 Byte)
Trainable params: 225 (900.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## 6-Model Derleme

In [19]:
model.compile(
    #binary cross entropy
    loss = "binary_crossentropy",

    #optimizer : Adam ve learning ratei otomatik adapte edip ağırlıkları günceller
    optimizer = "adam", 

    #metrik: accuracy çünkü sınıflandırma probleminde en temel performans metriğidir
    metrics = ["accuracy"]
    
)

#Sınıflandırme ve Regresyon Farkı:
#Sınıflandırma (classification): modelin çıktısının kategori veya sınıf olduğu problemleri ifade eder (örn: müşteri ayrılır / ayrılmaz).
#Regresyon (regression):modelin çıktısının sürekli sayısal bir değer olduğu problemleri ifade eder (örn: ev fiyatı tahmini veya satış miktarı).

## 7- Model Eğitimi

In [26]:
training_results = model.fit(X_train, y_train, epochs = 20, batch_size = 32, validation_data=(X_val, y_val))
#epochs -> eğitim kaç kez tekrar edecek
# batch_size -> her iterasyonda modele verilecek örnek sayısı

Epoch 1/20
53/53 [==============================] - 0s 759us/step - loss: 0.6882 - accuracy: 0.6935 - val_loss: 1.8552 - val_accuracy: 0.4778
Epoch 2/20
53/53 [==============================] - 0s 380us/step - loss: 0.8173 - accuracy: 0.6518 - val_loss: 0.5843 - val_accuracy: 0.7139
Epoch 3/20
53/53 [==============================] - 0s 359us/step - loss: 0.5899 - accuracy: 0.7089 - val_loss: 0.7392 - val_accuracy: 0.6306
Epoch 4/20
53/53 [==============================] - 0s 380us/step - loss: 0.6575 - accuracy: 0.6851 - val_loss: 0.6135 - val_accuracy: 0.6944
Epoch 5/20
53/53 [==============================] - 0s 376us/step - loss: 1.2209 - accuracy: 0.6167 - val_loss: 1.5793 - val_accuracy: 0.5944
Epoch 6/20
53/53 [==============================] - 0s 364us/step - loss: 0.9304 - accuracy: 0.6387 - val_loss: 0.7654 - val_accuracy: 0.6278
Epoch 7/20
53/53 [==============================] - 0s 360us/step - loss: 0.7284 - accuracy: 0.6780 - val_loss: 0.6577 - val_accuracy: 0.6722
Epoch 

## 8- Model Değerlendirme (Eval)

In [31]:
#test set üzerinde değerlendirme
#verbose=0 ekrana çıktı yazdırma demek
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose = 0)
print(f"Test loss: {test_loss:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print(" ")

#eğer training çok yüksek %90 ama val %70 falansa hafif overfitting
#eğer train /73, val 72 falansa ideal
#bizim modelimize bakınca modelin genelleme yapabildiğini görürüz
print("Last train acc:", round(training_results.history["accuracy"][-1], 3))
print("Last val acc:", round(training_results.history["val_accuracy"][-1], 3))

Test loss: 0.543
Test accuracy: 0.725
 
Last train acc: 0.685
Last val acc: 0.722


## 9- Tahmin yapma

In [35]:
#Yeni bir müşteri için churn tahmini yap
#[age, monthly_spend, subscription, support_request]
new_customer = np.array([[28, 1500, 6, 4]], dtype=np.float32)

#model tahmini
prediction = model.predict(new_customer)
probability = prediction[0][0]
print(f"Churn olasılığı: {probability:.4f}")

#olasılığa göre karar
if probability > 0.5:
    print("Müşteri ayrılabilir (churn = 1)")
else:
    print("Müşteri kalabilir (churn = 0)")
                                

1/1 [==============================] - 0s 9ms/step
Churn olasılığı: 0.0172
Müşteri kalabilir (churn = 0)


# Özet
- Model doğruluğu: %72 seviyesinde gerçekleşmiştir  
- Model, müşteri davranışlarını analiz ederek churn (müşteri ayrılması) riskini tahmin edebilmektedir  
- Yüksek riskli müşteriler erken aşamada belirlenebilir  
- Bu sayede kampanya, indirim veya müşteri destek stratejileri ile müşteri kaybı azaltılabilir